# Figure 8 source analysis — Monte Carlo annealing inference

This is the resumable Monte Carlo workflow that generated the archived fit table used by **Figure 8**. The calculation jointly fits the n- and p-InP relative-PL ratios at every annealing state while propagating shared physical and experimental nuisance parameters.

The supplied CSV has scientific configuration ID `cc3991e62a14ff65`. This notebook retains the historical `iii_v_bulk` v0.9.0 software identity in that hash because SURPASS v1.0.0 is the renamed release of the same model code. The runtime imports use SURPASS. Notebook outputs are cleared, and `RUN_TRAJECTORY` is `False` by default so opening the notebook does not start a multi-hour batch.


In [ ]:
import sys
import time
import os
import json
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.optimize import least_squares
from scipy.stats import qmc, spearmanr
from IPython.display import display
import matplotlib as mpl

root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/'src'/'surpass').exists()), None)
if root is None:
    raise FileNotFoundError('Could not locate the SURPASS project root.')
sys.path.insert(0, str(root/'src'))

import surpass
from surpass import (
    BulkModel, BulkTransportTable, InterfaceChargeModel, InterfaceDefectModel,
    Layer, OpticalStack, SteadyState1DSolver, SurfaceBoundaryTable,
    SurfaceSRHModel, Wafer, get_optical_material,
)

try:
    import scienceplots
    plt.style.use(['science', 'notebook', 'no-latex'])
except ImportError:
    plt.style.use('default')

plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 300, 'font.size': 10})
output_dir = Path.cwd()/'output'
output_dir.mkdir(parents=True, exist_ok=True)

print('SURPASS version:', surpass.__version__)
print('loaded from:', Path(surpass.__file__).resolve())


## 1. Fixed experiment and numerical model

The optical stack and material assumptions are inherited from the endpoint
notebook. Annealing is assumed not to change the layer thicknesses or optical
constants. If ellipsometry shows an appreciable change, those optical changes
should be added as state-dependent inputs.


In [ ]:
wavelength_nm = 514.0
thickness_um = 620.0
thickness_cm = thickness_um*1e-4
rear_s_cm_s = 1e5

N_D_NOMINAL = 8.5e18
N_A_NOMINAL = 5.1e18
INTENSITY_N_NOMINAL = 1.0
INTENSITY_P_NOMINAL = 13.0

RATIO_N = 2.826
RATIO_P = 0.5969

# Adamowicz et al., Vacuum 63, 223-227 (2001), DOI:
# 10.1016/S0042-207X(01)00195-6. These are literature-motivated
# starting values for chemically polished InP(100), not measured POx/InP values.
SIGMA_N_NOMINAL = 1e-14
SIGMA_P_NOMINAL = 1e-13
VTH_N = 4.13e7
VTH_P = 1.51e7

ENABLE_BGN = True
INCLUDE_INTERFACE_CHARGE = True
INTERFACE_CHARGE = InterfaceChargeModel(0.5, 0.02) if INCLUDE_INTERFACE_CHARGE else None

air = get_optical_material('air')
inp_optical = get_optical_material('InP_demo')
pox_optical = get_optical_material('POx_demo')
alox_optical = get_optical_material('AlOx_demo')
coated_pump = OpticalStack(
    air,
    [Layer(alox_optical, 13.0, 'AlOx'), Layer(pox_optical, 4.0, 'POx')],
    inp_optical,
)
bare_pump = OpticalStack(air, [], inp_optical)

# Slightly reduced mesh/table sizes make repeated inverse solutions practical.
# Increase these for the final convergence check.
edges_cm = np.r_[0.0, np.geomspace(1e-8, 5e-4, 80),
                 np.linspace(5e-4, thickness_cm, 66)[1:]]
TRANSPORT_POINTS = 121
BOUNDARY_POINTS = 61
ENERGY_POINTS = 101
MAX_NFEV_TRANSPORT = 900


## 2. Enter the annealing-series PL ratios here

`ratio_n` and `ratio_p` must both be normalized to the same corresponding bare
n-InP and p-InP measurements. The four supplied experimental states are entered
below. The numerical temperatures are treated as authoritative; consequently,
the two intermediate states are labelled 100 °C and 150 °C.

The `independent_rel_sigma` columns describe state-specific numerator scatter.
Uncertainty shared through the common bare denominator is configured separately
below. If your quoted ratio uncertainties already include the complete ratio
uncertainty, set the common bare uncertainty to zero to avoid double counting.


In [ ]:
annealing_data = pd.DataFrame([
    # state             T (C)   n ratio  p ratio  independent relative 1-sigma
    ('As-deposited',     25.0,  0.91987,   0.74597,  0.05, 0.05),
    ('150 C anneal',    150.0,  1.286678,   0.6994,  0.05, 0.05),
    ('200 C anneal',    200.0,  1.775588,   0.6566,  0.05, 0.05),
    ('250 C anneal',    250.0,  2.82558,   0.59693,  0.05, 0.05),
    ('300 C anneal',    300.0,  1.84854,   0.5773,  0.05, 0.05),
    ('350 C anneal',    350.0,  1.1879,   0.5561,  0.05, 0.05),
], columns=[
    'state', 'temperature_c', 'ratio_n', 'ratio_p',
    'independent_rel_sigma_n', 'independent_rel_sigma_p',
])

# Optional uncertainty shared by every ratio through the same bare denominator.
# These are set to zero until an experimental estimate is supplied.
COMMON_BARE_REL_SIGMA_N = 0.00
COMMON_BARE_REL_SIGMA_P = 0.00

display(annealing_data)

if annealing_data.state.duplicated().any():
    raise ValueError('Every annealing state name must be unique.')
if (annealing_data[['ratio_n','ratio_p']] <= 0).any().any():
    raise ValueError('All PL ratios must be positive.')


## 3. Shared uncertainty model

Every draw samples one common set of bulk, optical and interface-capture
assumptions. It also samples one common bare-PL error for each doping polarity
and an independent measurement error for every annealing state. The same
nuisance draw is retained while fitting the complete trajectory.


In [ ]:
uncertainty = pd.DataFrame([
    ('bulk SRH lifetime', 'log-uniform', 2e-9, 10e-9, 's'),
    ('photon recycling probability', 'uniform', 0.0, 0.93, '-'),
    ('bare Dit', 'log-uniform', 3e11, 3e12, 'eV^-1 cm^-2'),
    ('bare fixed charge', 'uniform', -5e11, 5e11, 'cm^-2'),
    ('capture cross-section scale', 'log-uniform', 10**-0.5, 10**0.5, 'x nominal'),
    ('sigma_p/sigma_n multiplier', 'log-uniform', 10**-0.5, 10**0.5, 'x nominal ratio'),
    ('n-InP donor density', 'uniform', 0.9, 1.1, 'x nominal'),
    ('p-InP acceptor density', 'uniform', 0.9, 1.1, 'x nominal'),
    ('n-InP intensity', 'uniform', 0.9, 1.1, 'x nominal'),
    ('p-InP intensity', 'uniform', 0.9, 1.1, 'x nominal'),
], columns=['quantity','distribution','low','high','unit or note'])
display(uncertainty)

def log_uniform(u, low, high):
    return 10**(np.log10(low) + u*(np.log10(high)-np.log10(low)))

def make_trajectory_draws(n, seed, data):
    m = len(data)
    # 10 physical nuisance variables + 2 shared denominator errors
    # + 2 independent numerator errors per state.
    u = qmc.LatinHypercube(d=12 + 2*m, seed=seed).random(n)
    from scipy.stats import norm
    z = norm.ppf(np.clip(u, 0.005, 0.995))

    scale = log_uniform(u[:,3], 10**-0.5, 10**0.5)
    ratio_multiplier = log_uniform(u[:,4], 10**-0.5, 10**0.5)
    frame = pd.DataFrame({
        'tau_srh_s': log_uniform(u[:,0], 2e-9, 10e-9),
        'p_recycling': 0.93*u[:,1],
        'bare_dit': log_uniform(u[:,2], 3e11, 3e12),
        'sigma_n': SIGMA_N_NOMINAL*scale/np.sqrt(ratio_multiplier),
        'sigma_p': SIGMA_P_NOMINAL*scale*np.sqrt(ratio_multiplier),
        'donor_cm3': N_D_NOMINAL*(0.9 + 0.2*u[:,5]),
        'acceptor_cm3': N_A_NOMINAL*(0.9 + 0.2*u[:,6]),
        'intensity_n': INTENSITY_N_NOMINAL*(0.9 + 0.2*u[:,7]),
        'intensity_p': INTENSITY_P_NOMINAL*(0.9 + 0.2*u[:,8]),
        'bare_qf_cm2': -5e11 + 1e12*u[:,9],
        'z_bare_n': z[:,10],
        'z_bare_p': z[:,11],
    })
    for j, state in enumerate(data.state):
        frame[f'z_num_n_{j}'] = z[:,12+j]
        frame[f'z_num_p_{j}'] = z[:,12+m+j]
    return frame

def sampled_target_ratios(draw, state_index, state_row):
    # Ratio = numerator / common bare denominator.
    target_n = state_row.ratio_n*np.exp(
        state_row.independent_rel_sigma_n*draw[f'z_num_n_{state_index}']
        - COMMON_BARE_REL_SIGMA_N*draw.z_bare_n)
    target_p = state_row.ratio_p*np.exp(
        state_row.independent_rel_sigma_p*draw[f'z_num_p_{state_index}']
        - COMMON_BARE_REL_SIGMA_P*draw.z_bare_p)
    return np.array([target_n, target_p], dtype=float)


## 4. One shared forward model per Monte Carlo draw

The bare solutions, bulk transport tables and generation profiles are calculated
only once per draw. The cache of coated-interface solutions is shared across all
annealing states, which makes the trajectory substantially faster than running
independent endpoint notebooks.


In [ ]:
def _surface_model(wafer, charge_cm2, dit, sigma_n, sigma_p):
    defects = InterfaceDefectModel(
        dit, sigma_n, sigma_p, VTH_N, VTH_P,
        shape='constant', energy_points=ENERGY_POINTS,
    )
    return SurfaceSRHModel(
        BulkModel(wafer, enable_bgn=ENABLE_BGN), charge_cm2, defects,
        electrostatics='self_consistent', driving_force='excess_product',
        interface_charge=INTERFACE_CHARGE,
    )

class TrajectoryForwardModel:
    def __init__(self, row):
        self.row = row
        self.labels = ('n-InP', 'p-InP')
        self.parts = {}
        settings = {
            'n-InP': (Wafer('InP', donor_cm3=row.donor_cm3,
                            thickness_um=thickness_um), row.intensity_n),
            'p-InP': (Wafer('InP', acceptor_cm3=row.acceptor_cm3,
                            thickness_um=thickness_um), row.intensity_p),
        }
        for label, (wafer, intensity) in settings.items():
            bulk = BulkModel(wafer, enable_bgn=ENABLE_BGN,
                             tau_srh_s=row.tau_srh_s)
            transport = BulkTransportTable.from_bulk_model(
                bulk, 1.0, 1e21, points=TRANSPORT_POINTS)
            solver = SteadyState1DSolver(transport)
            coated_generation = coated_pump.generation_cell_average_cm3_s(
                wavelength_nm, intensity, edges_cm)
            bare_generation = bare_pump.generation_cell_average_cm3_s(
                wavelength_nm, intensity, edges_cm)
            bare_surface = _surface_model(
                wafer, row.bare_qf_cm2, row.bare_dit,
                row.sigma_n, row.sigma_p)
            bare_boundary = SurfaceBoundaryTable.from_surface_model(
                bare_surface, 1.0, 1e21, points=BOUNDARY_POINTS)
            bare_result = solver.solve(
                thickness_cm, bare_generation, bare_boundary, rear_s_cm_s,
                cell_edges_cm=edges_cm, max_nfev=MAX_NFEV_TRANSPORT,
                recycling_probability=row.p_recycling)
            if not bare_result.converged:
                raise RuntimeError(f'Bare reference did not converge for {label}')
            self.parts[label] = (
                wafer, solver, coated_generation,
                bare_result.internal_radiative_emission_cm2_s,
            )
        self.cache = {}

    def ratios(self, log10_dit, q12):
        key = (round(float(log10_dit), 12), round(float(q12), 12))
        if key in self.cache:
            return self.cache[key]
        dit = 10**float(log10_dit)
        charge = float(q12)*1e12
        values = []
        for label in self.labels:
            wafer, solver, generation, bare_pl = self.parts[label]
            surface = _surface_model(
                wafer, charge, dit, self.row.sigma_n, self.row.sigma_p)
            boundary = SurfaceBoundaryTable.from_surface_model(
                surface, 1.0, 1e21, points=BOUNDARY_POINTS)
            result = solver.solve(
                thickness_cm, generation, boundary, rear_s_cm_s,
                cell_edges_cm=edges_cm, max_nfev=MAX_NFEV_TRANSPORT,
                recycling_probability=self.row.p_recycling)
            values.append(result.internal_radiative_emission_cm2_s/bare_pl
                          if result.converged else np.nan)
        self.cache[key] = np.asarray(values)
        return self.cache[key]

    def residual(self, x, targets):
        pred = self.ratios(x[0], x[1])
        if np.any(~np.isfinite(pred)) or np.any(pred <= 0):
            return np.array([1e3, 1e3])
        return np.log(pred/targets)


## 5. Fit every state without imposing a trend

The previous state's solution is used as an efficient warm start, but remote
starts are retained to detect another solution branch. A fit is classified by
its joint logarithmic mismatch in the two PL ratios.


In [ ]:
# Standard search region used for the annealed states
LOGDIT_BOUNDS = (9.5, 12.8)
Q12_BOUNDS = (-1.0, 6.0)

BASE_STARTS = [
    (10.5, 1.0),
    (10.5, 2.5),
    (11.5, 4.0),
]

# Wider search region for the incompatible as-deposited state
AS_DEP_LOGDIT_BOUNDS = (9.0, 14.5)
AS_DEP_Q12_BOUNDS = (-10.0, 10.0)

AS_DEP_STARTS = [
    (10.5,  1.0),
    (11.5,  0.0),
    (12.5, -1.0),
    (13.0, -3.0),
    (13.5, -6.0),
    (14.0, -9.0),
    (13.5,  2.0),
    (14.0,  6.0),
]

EXACT_MATCH_TOL = 0.025

def solve_state(
    model,
    targets,
    previous_solution=None,
    logdit_bounds=LOGDIT_BOUNDS,
    q12_bounds=Q12_BOUNDS,
    base_starts=BASE_STARTS,
):
    t0 = time.perf_counter()
    starts = []

    # Only use the previous solution if it lies inside the current bounds.
    if (
        previous_solution is not None and
        np.all(np.isfinite(previous_solution)) and
        logdit_bounds[0] <= previous_solution[0] <= logdit_bounds[1] and
        q12_bounds[0] <= previous_solution[1] <= q12_bounds[1]
    ):
        starts.append(tuple(previous_solution))

    starts.extend(base_starts)

    # Preserve order while removing duplicate starts.
    starts = list(dict.fromkeys(
        (round(a, 8), round(b, 8))
        for a, b in starts
    ))

    try:
        candidates = []

        for x0 in starts:
            fit = least_squares(
                lambda x: model.residual(x, targets),
                x0=x0,
                bounds=(
                    [logdit_bounds[0], q12_bounds[0]],
                    [logdit_bounds[1], q12_bounds[1]],
                ),
                xtol=2e-4,
                ftol=2e-4,
                gtol=2e-4,
                max_nfev=35,
                x_scale=[1.0, 2.0],
                diff_step=[2e-3, 2e-3],
            )

            candidates.append(fit)

        fit = min(
            candidates,
            key=lambda candidate: np.sum(candidate.fun**2)
        )

        pred = model.ratios(*fit.x)
        residuals = model.residual(fit.x, targets)

        at_boundary = (
            abs(fit.x[0] - logdit_bounds[0]) < 0.02 or
            abs(fit.x[0] - logdit_bounds[1]) < 0.02 or
            abs(fit.x[1] - q12_bounds[0]) < 0.03 or
            abs(fit.x[1] - q12_bounds[1]) < 0.03
        )

        return {
            'success': bool(
                fit.success and np.all(np.isfinite(pred))
            ),
            'log10_dit_fit': fit.x[0],
            'dit_fit': 10**fit.x[0],
            'qf_over_q_fit_cm2': fit.x[1] * 1e12,
            'pred_ratio_n': pred[0],
            'pred_ratio_p': pred[1],
            'log_residual_n': residuals[0],
            'log_residual_p': residuals[1],
            'rms_log_residual': np.sqrt(
                np.mean(residuals**2)
            ),
            'at_boundary': at_boundary,
            'n_cached_forward_points': len(model.cache),
            'seconds': time.perf_counter() - t0,
            'message': fit.message,
        }

    except Exception as exc:
        return {
            'success': False,
            'log10_dit_fit': np.nan,
            'dit_fit': np.nan,
            'qf_over_q_fit_cm2': np.nan,
            'pred_ratio_n': np.nan,
            'pred_ratio_p': np.nan,
            'log_residual_n': np.nan,
            'log_residual_p': np.nan,
            'rms_log_residual': np.inf,
            'at_boundary': False,
            'n_cached_forward_points': len(model.cache),
            'seconds': time.perf_counter() - t0,
            'message': repr(exc),
        }


## 6. Resumable trajectory batches

Every completed draw/state fit is written atomically. To resume an interrupted
batch, retain its ID, seed and size. To add statistically new trajectories, use
a new batch ID and seed rather than increasing an existing Latin-hypercube size.


In [ ]:
RUN_TRAJECTORY = False
BATCH_ID = 'trajectory_006'
BATCH_SEED = 20261003
N_DRAWS_IN_BATCH = 6
INFERENCE_CONFIG_VERSION = 2

csv_path = (
    root / 'notebooks' / 'paper_inp_pox_alox' / 'data' /
    'inp_annealing_trajectory_batched_wide_asdep.csv'
)

if annealing_data[['ratio_n','ratio_p']].isna().any().any():
    missing_states = annealing_data.loc[
        annealing_data[['ratio_n','ratio_p']].isna().any(axis=1), 'state'].tolist()
    raise ValueError(
        'Enter both experimental PL ratios before running. Missing: '
        + ', '.join(missing_states))

# Preserve the software identity used when the archived configuration ID was created.
# SURPASS v1.0.0 is the renamed public release of this model code.
SOURCE_MODEL_VERSION = '0.9.0'
ARCHIVED_CONFIG_ID = 'cc3991e62a14ff65'

scientific_config = {
    'version': INFERENCE_CONFIG_VERSION,
    'iii_v_bulk_version': SOURCE_MODEL_VERSION,
    'annealing_data': annealing_data.to_dict(orient='records'),
    'common_bare_rel_sigma_n': COMMON_BARE_REL_SIGMA_N,
    'common_bare_rel_sigma_p': COMMON_BARE_REL_SIGMA_P,
    'tau_srh_range_s': [2e-9,10e-9],
    'photon_recycling_range': [0.0,0.93],
    'bare_dit_range': [3e11,3e12],
    'bare_qf_range_cm2': [-5e11,5e11],
    'capture_scale_range': [10**-0.5,10**0.5],
    'capture_ratio_range': [10**-0.5,10**0.5],
    'logdit_bounds': list(LOGDIT_BOUNDS),
    'q12_bounds': list(Q12_BOUNDS),
    'starts': [list(x) for x in BASE_STARTS],
    'exact_match_tol': EXACT_MATCH_TOL,
    'enable_bgn': ENABLE_BGN,
    'include_interface_charge': INCLUDE_INTERFACE_CHARGE,
    'mesh_edges': len(edges_cm),
    'transport_points': TRANSPORT_POINTS,
    'boundary_points': BOUNDARY_POINTS,
    'energy_points': ENERGY_POINTS,
    'as_deposited_logdit_bounds': list(AS_DEP_LOGDIT_BOUNDS),
    'as_deposited_q12_bounds': list(AS_DEP_Q12_BOUNDS),
    'as_deposited_starts': [
        list(x) for x in AS_DEP_STARTS
    ],
}
config_json = json.dumps(scientific_config, sort_keys=True, separators=(',',':'))
CONFIG_ID = hashlib.sha256(config_json.encode()).hexdigest()[:16]
if CONFIG_ID != ARCHIVED_CONFIG_ID:
    raise RuntimeError(
        f'Configuration drift: computed {CONFIG_ID}, expected {ARCHIVED_CONFIG_ID}.')
print('Scientific configuration ID:', CONFIG_ID, '(matches archived CSV)')

def atomic_save(frame, destination):
    temporary = Path(destination).with_suffix(Path(destination).suffix+'.tmp')
    frame.to_csv(temporary, index=False)
    os.replace(temporary, destination)

def classify(frame):
    frame = frame.copy()
    frame['usable_best_fit'] = frame.success.astype(bool) & ~frame.at_boundary.astype(bool)
    frame['compatible'] = frame.usable_best_fit & (frame.rms_log_residual <= EXACT_MATCH_TOL)
    return frame

existing = pd.read_csv(csv_path) if csv_path.exists() else pd.DataFrame()
if not existing.empty:
    required = {'config_id','batch_id','batch_seed','batch_size','draw_id','state'}
    missing = sorted(required-set(existing.columns))
    if missing:
        raise ValueError(f'Existing trajectory CSV lacks resumable metadata: {missing}')
    if set(existing.config_id.astype(str)) != {CONFIG_ID}:
        raise ValueError(
            'Existing results were generated for different data/model settings. '
            'Use a new csv_path or restore the previous configuration.')

draws = make_trajectory_draws(N_DRAWS_IN_BATCH,BATCH_SEED,annealing_data)
draws['config_id'] = CONFIG_ID
draws['batch_id'] = str(BATCH_ID)
draws['batch_seed'] = int(BATCH_SEED)
draws['batch_size'] = int(N_DRAWS_IN_BATCH)
draws['draw_in_batch'] = np.arange(N_DRAWS_IN_BATCH,dtype=int)
draws['draw_id'] = [f'{CONFIG_ID}:{BATCH_ID}:{i:05d}' for i in range(N_DRAWS_IN_BATCH)]

if not existing.empty:
    same_batch = existing.batch_id.astype(str).eq(str(BATCH_ID))
    if same_batch.any():
        seeds = set(pd.to_numeric(existing.loc[same_batch,'batch_seed']).astype(int))
        sizes = set(pd.to_numeric(existing.loc[same_batch,'batch_size']).astype(int))
        if seeds != {int(BATCH_SEED)} or sizes != {int(N_DRAWS_IN_BATCH)}:
            raise ValueError('Existing batch ID has a different seed or size.')
    reused_seed = pd.to_numeric(existing.batch_seed,errors='coerce').eq(BATCH_SEED) & ~same_batch
    if reused_seed.any():
        raise ValueError('This seed is already used by another batch; choose a new seed.')

mc = existing.copy()
completed_pairs = (set(zip(mc.draw_id.astype(str),mc.state.astype(str)))
                   if not mc.empty else set())

if RUN_TRAJECTORY:
    for draw_number,(_,draw) in enumerate(draws.iterrows(),start=1):
        states_missing = [s for s in annealing_data.state
                          if (draw.draw_id,str(s)) not in completed_pairs]
        if not states_missing:
            print(f'Draw {draw_number}/{len(draws)} already complete; skipped.')
            continue
        model = TrajectoryForwardModel(draw)
        previous = None
        # Recover the latest earlier fitted state when resuming mid-trajectory.
        for state_index,state_row in annealing_data.iterrows():
            prior = (mc.loc[(mc.draw_id.astype(str)==str(draw.draw_id)) &
                            (mc.state.astype(str)==str(state_row.state))]
                     if not mc.empty else pd.DataFrame())
            if not prior.empty and bool(prior.iloc[-1].success):
                previous = prior.iloc[-1][['log10_dit_fit','qf_over_q_fit_cm2']].to_numpy(float)
                previous[1] /= 1e12
                continue
            targets = sampled_target_ratios(draw,state_index,state_row)
            if state_row.state == 'As-deposited':
                result = solve_state(
                    model,
                    targets,
                    previous_solution=None,
                    logdit_bounds=AS_DEP_LOGDIT_BOUNDS,
                    q12_bounds=AS_DEP_Q12_BOUNDS,
                    base_starts=AS_DEP_STARTS,
                    )
            else:
                result = solve_state(
                    model,
                    targets,
                    previous_solution=previous,
                    logdit_bounds=LOGDIT_BOUNDS,
                    q12_bounds=Q12_BOUNDS,
                    base_starts=BASE_STARTS,
                )
            record = {
                **draw.to_dict(), **result,
                'state_index': int(state_index), 'state': state_row.state,
                'temperature_c': state_row.temperature_c,
                'observed_ratio_n': state_row.ratio_n,
                'observed_ratio_p': state_row.ratio_p,
                'target_ratio_n': targets[0], 'target_ratio_p': targets[1],
            }
            row_frame = classify(pd.DataFrame([record]))
            mc = pd.concat([mc,row_frame],ignore_index=True)
            mc = mc.drop_duplicates(['draw_id','state'],keep='last').reset_index(drop=True)
            atomic_save(mc,csv_path)
            completed_pairs.add((str(draw.draw_id),str(state_row.state)))
            # Only compatible solutions are suitable as warm starts.
            # In particular, do not propagate the incompatible as-deposited
            # result into the first annealed-state fit.
            if (
                result['success'] and
                not result['at_boundary'] and
                result['rms_log_residual'] <= EXACT_MATCH_TOL
            ):
                previous = np.array([
                    result['log10_dit_fit'],
                    result['qf_over_q_fit_cm2'] / 1e12
                ])
            else:
                previous = None
            print(
                f'draw {draw_number}/{len(draws)}, state {state_index+1}/{len(annealing_data)} '
                f'({state_row.state}): Dit={result["dit_fit"]:.3g}, '
                f'Qf/q={result["qf_over_q_fit_cm2"]:.3g}, '
                f'rms={result["rms_log_residual"]:.3g}, '
                f'{result["seconds"]:.1f} s [checkpoint saved]')

if mc.empty:
    raise RuntimeError('No trajectory results are available.')
mc = classify(mc)
display(mc[['batch_id','draw_in_batch','state','compatible','dit_fit',
            'qf_over_q_fit_cm2','pred_ratio_n','pred_ratio_p',
            'rms_log_residual','seconds']])
print(f'{len(mc)} fitted draw/state pairs saved in {csv_path}.')


## 7. Compatible paired trajectories

For trend statistics, retain only Monte Carlo draws with compatible solutions
at every state listed in `TREND_STATES`. The preliminary validation identifies
the as-deposited condition separately because it is not reproduced by the
current two-parameter surface model. This avoids discarding the well-defined
annealing trend while keeping the as-deposited model mismatch visible.


In [ ]:
state_order = annealing_data.state.tolist()

# Preliminary five-draw validation showed that the as-deposited pair cannot be
# reproduced within the current two-parameter surface model, whereas all three
# annealed states generally can. Keep the as-deposited diagnostics, but do not
# let that known model mismatch erase the annealing trend among compatible states.
TREND_STATES = ['150 C anneal','200 C anneal','250 C anneal','300 C anneal','350 C anneal']
missing_trend_states = sorted(set(TREND_STATES)-set(state_order))
if missing_trend_states:
    raise ValueError(f'TREND_STATES are missing from annealing_data: {missing_trend_states}')

mc['state'] = pd.Categorical(mc.state,categories=state_order,ordered=True)
trend_candidates = mc.loc[mc.state.isin(TREND_STATES)].copy()
compatibility = trend_candidates.groupby('draw_id',observed=True).agg(
    n_states=('state','nunique'), all_compatible=('compatible','all'))
complete_ids = compatibility.index[
    (compatibility.n_states==len(TREND_STATES)) & compatibility.all_compatible]
paired = trend_candidates.loc[trend_candidates.draw_id.isin(complete_ids)].copy()
print(f'Complete compatible annealed trajectories: {len(complete_ids)} of '
      f'{mc.draw_id.nunique()} saved draws.')

as_deposited_diagnostic = mc.loc[mc.state=='As-deposited',[
    'draw_id','target_ratio_n','target_ratio_p','pred_ratio_n','pred_ratio_p',
    'dit_fit','qf_over_q_fit_cm2','rms_log_residual','compatible']]
display(as_deposited_diagnostic)

summary_rows = []
for state in TREND_STATES:
    subset = paired.loc[paired.state==state]
    for quantity in ['dit_fit','qf_over_q_fit_cm2']:
        values = subset[quantity]
        summary_rows.append({
            'state':state,'quantity':quantity,'n':len(values),
            'p2.5':values.quantile(.025),'p16':values.quantile(.16),
            'median':values.median(),'p84':values.quantile(.84),
            'p97.5':values.quantile(.975),
        })
trajectory_summary = pd.DataFrame(summary_rows)
display(trajectory_summary)


## 8. Publication-oriented trajectory plots

The uncertainty bands describe the specified model/measurement ensemble, not a
standalone electrical measurement of $D_{it}$ or $Q_f$. The $D_{it}$–$Q_f$
panel shows the median evolution without imposing monotonicity.


In [ ]:
if len(complete_ids) < 3:
    print('Too few complete compatible trajectories for stable plots.')

else:
    SHOW_AS_DEPOSITED = True
    JITTER_WIDTH = 0.09
    RMS_REFERENCE = EXACT_MATCH_TOL

    rng_plot = np.random.default_rng(20260930)

    annealed_colors = {
        '150 C anneal': 'tab:purple',
        '200 C anneal': 'tab:blue',
        '250 C anneal': 'tab:green',
        '300 C anneal': 'tab:orange',
        '350 C anneal': 'tab:red',
    }

    as_deposited_color = '0.35'

    display_labels = {
        'As-deposited': 'As-deposited',
        '150 C anneal': r'150 $^\circ$C',
        '200 C anneal': r'200 $^\circ$C',
        '250 C anneal': r'250 $^\circ$C',
        '300 C anneal': r'300 $^\circ$C',
        '350 C anneal': r'350 $^\circ$C',
    }

    if SHOW_AS_DEPOSITED:
        plot_states = ['As-deposited'] + TREND_STATES
    else:
        plot_states = TREND_STATES

    state_positions = {
        state: position
        for position, state in enumerate(plot_states)
    }

    fig, axes = plt.subplots(
        2, 1,
        figsize=(7.2, 7.2),
        sharex=True,
        constrained_layout=True,
    )


    def plot_parameter(
        ax,
        quantity,
        ylabel,
        scale=1.0,
        log=False,
    ):
        # --------------------------------------------------------
        # Individual compatible annealed solutions
        # --------------------------------------------------------
        for state in TREND_STATES:
            subset = paired.loc[
                paired.state == state
            ].copy()

            sizes, alphas = marker_properties(
                subset.rms_log_residual
            )

            jitter = rng_plot.uniform(
                -JITTER_WIDTH,
                JITTER_WIDTH,
                size=len(subset)
            )

            color = annealed_colors[state]

            rgba = np.tile(
                mpl.colors.to_rgba(color),
                (len(subset), 1)
            )
            rgba[:, 3] = alphas

            ax.scatter(
                state_positions[state] + jitter,
                subset[quantity] / scale,
                s=sizes,
                c=rgba,
                edgecolors='none',
                zorder=2,
            )

        # --------------------------------------------------------
        # Incompatible as-deposited best fits
        # --------------------------------------------------------
        if SHOW_AS_DEPOSITED:
            as_dep = mc.loc[
                (mc.state == 'As-deposited') &
                mc.success.astype(bool) &
                np.isfinite(mc[quantity])
            ].copy()

            if len(as_dep):
                sizes, alphas = marker_properties(
                    as_dep.rms_log_residual
                )

                jitter = rng_plot.uniform(
                    -JITTER_WIDTH,
                    JITTER_WIDTH,
                    size=len(as_dep)
                )

                rgba = np.tile(
                    mpl.colors.to_rgba(
                        as_deposited_color
                    ),
                    (len(as_dep), 1)
                )
                rgba[:, 3] = alphas

                ax.scatter(
                    state_positions['As-deposited'] + jitter,
                    as_dep[quantity] / scale,
                    s=sizes,
                    facecolors='none',
                    edgecolors=rgba,
                    linewidths=1.0,
                    zorder=2,
                )

        # --------------------------------------------------------
        # Annealed-state medians and 16–84% intervals
        # --------------------------------------------------------
        summary = percentile_summary(
            paired,
            quantity,
            TREND_STATES
        )

        for state in TREND_STATES:
            row = summary.loc[state]

            median = row['median'] / scale
            lower = (
                row['median'] - row['p16']
            ) / scale
            upper = (
                row['p84'] - row['median']
            ) / scale

            ax.errorbar(
                state_positions[state],
                median,
                yerr=np.array([[lower], [upper]]),

                # Black summary symbols and error bars
                fmt='D',
                ms=6,
                color='black',
                markerfacecolor='black',
                markeredgecolor='white',
                markeredgewidth=0.6,
                ecolor='black',
                elinewidth=1.5,
                capsize=4,
                capthick=1.5,
                zorder=4,
            )

        if log:
            ax.set_yscale('log')

        ax.set_ylabel(ylabel)
        ax.grid(axis='y', alpha=.18)


    # ------------------------------------------------------------
    # Top: effective fixed charge
    # ------------------------------------------------------------
    plot_parameter(
        axes[0],
        quantity='qf_over_q_fit_cm2',
        ylabel=r'$Q_f/q$ ($10^{12}$ cm$^{-2}$)',
        scale=1e12,
    )

    axes[0].axhline(
        0,
        color='.55',
        lw=.8,
        ls=':',
        zorder=1,
    )

    axes[0].set_title(
        'Inferred effective fixed charge'
    )


    # ------------------------------------------------------------
    # Bottom: interface-state density
    # ------------------------------------------------------------
    plot_parameter(
        axes[1],
        quantity='dit_fit',
        ylabel=(
            r'$D_\mathrm{it}$ '
            r'(eV$^{-1}$ cm$^{-2}$)'
        ),
        log=True,
    )

    axes[1].set_title(
        'Inferred interface-state density'
    )


    # ------------------------------------------------------------
    # Shared x-axis
    # ------------------------------------------------------------
    positions = np.arange(len(plot_states))

    axes[1].set_xticks(
        positions,
        [display_labels[state] for state in plot_states],
        rotation=0,
        ha='center',
    )

    axes[1].set_xlabel('Annealing condition')

    fig.savefig(
        output_dir /
        'paper_inp_annealing_inferred_parameters.png',
        bbox_inches='tight',
    )

## 9. Paired changes and trend probabilities

The first entered state is used as the reference. Because each difference is
formed within one nuisance draw, common model uncertainties partly cancel.
No monotonic trend is assumed; the probabilities below are empirical fractions
of complete compatible trajectories.


In [ ]:
if len(complete_ids):
    reference_state = TREND_STATES[0]
    ref = paired.loc[paired.state==reference_state].set_index('draw_id')
    paired['delta_log10_dit'] = (
        paired.log10_dit_fit - paired.draw_id.map(ref.log10_dit_fit))
    paired['delta_qf_cm2'] = (
        paired.qf_over_q_fit_cm2 - paired.draw_id.map(ref.qf_over_q_fit_cm2))

    trend_rows=[]
    for state in TREND_STATES[1:]:
        subset=paired.loc[paired.state==state]
        trend_rows.append({
            'state':state,
            'P(Qf > reference)':np.mean(subset.delta_qf_cm2>0),
            'P(Dit < reference)':np.mean(subset.delta_log10_dit<0),
            'median delta Qf/q (cm^-2)':subset.delta_qf_cm2.median(),
            'median Dit ratio to reference':10**subset.delta_log10_dit.median(),
        })
    trend_probabilities=pd.DataFrame(trend_rows)
    display(trend_probabilities)

    # Strict monotonicity is evaluated only as an outcome, never imposed.
    wide_q=paired.pivot(index='draw_id',columns='state',values='qf_over_q_fit_cm2')[TREND_STATES]
    wide_d=paired.pivot(index='draw_id',columns='state',values='log10_dit_fit')[TREND_STATES]
    print('Fraction with strictly increasing Qf:',
          np.mean(np.all(np.diff(wide_q.to_numpy(),axis=1)>0,axis=1)))
    print('Fraction with strictly decreasing Dit:',
          np.mean(np.all(np.diff(wide_d.to_numpy(),axis=1)<0,axis=1)))


## 10. Interpretation safeguards

- Report the number of complete compatible trajectories and the rejection rate.
- Inspect high-RMS states before excluding them; rerun representative cases with
  more starts or a diagnostic grid to distinguish optimization failures from
  genuinely incompatible nuisance draws.
- Treat $Q_f$ as an effective sheet charge within the model. Its absolute value
  remains conditional on the assumed bare-surface charge.
- Emphasize paired changes with annealing; they are generally more robust than
  the absolute inferred parameters.
- If annealing changes optical constants, layer thickness, bulk lifetime or the
  PL collection efficiency, add those as state-dependent effects rather than
  forcing the change into $D_{it}$ and $Q_f$.
